# B-Scan Migration Playground – Subwavelength PSF **Vertical** TimeLapse Analysis

**Purpose:** Quantify the detection limit of **vertical** (depth) subwavelength scatterer movement using three migration methods.

A single PEC cylinder at **x = 2.0 m** (centre of the survey domain) sinks downward from its baseline depth (y = 0.224 m in gprMax coordinates, i.e. 0.676 m below the ice surface) by fractions of the wavelength: 1λ, ½λ, ¼λ, ⅛λ, ¹⁄₁₆λ, ¹⁄₃₂λ.  The 2λ scenario is omitted because it would push the scatterer below the domain floor.

Methods tested:
- **Kirchhoff** delay-and-sum (PyLops zero-offset operator)
- **Gazdag** phase-shift (f-k domain, exploding-reflector half-velocity)
- **Back-propagation** time-reversal (gprMax re-injection)

Each method produces a migrated image; the *time-lapse difference* (monitor − baseline) is analysed via a **vertical PSF profile** at x = 2.0 m.


# Input File Creation

In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import pyvista as pv
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot

import sys as _sys, pathlib as _pathlib
if str(_pathlib.Path.cwd()) not in _sys.path:
    _sys.path.insert(0, str(_pathlib.Path.cwd()))
from helper_functions.visualisation import (
    plot_domain_geometry, plot_bscan_grid, plot_migrated_grid, plot_wavefield_grid,
    plot_method_comparison_grid, plot_psf_grid, plot_spectrum_grid, plot_trace_comparison,
)


In [ ]:
# ── Standard figure auto-save (protocol: .wiki/FIGURES_PROTOCOL.md) ─────────
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
from helper_functions.figures import setup_autosave
setup_autosave(study="Vertical_TimeLapse_Study", prefix="VTL_")


In [ ]:
# Medium
eps_r      = 3.15
v_ice      = 0.299792458 / np.sqrt(eps_r)   # m/ns ≈ 0.16892
f_c_GHz    = 1.5                              # centre frequency [GHz]
wavelength = v_ice / f_c_GHz                  # m ≈ 0.1126
v_mig      = v_ice / 2                        # exploding-reflector half-velocity [m/ns]
t0_ns      = np.sqrt(2) / f_c_GHz            # Ricker peak delay [ns] ≈ 0.943

# Survey geometry
domain_x   = 4.0
n_traces   = 380
trace_step = 0.01   # m
rx_offset  = 0.1    # m (source-receiver offset in .in file)
x_traces   = (0.1 + rx_offset / 2) + np.arange(n_traces) * trace_step   # midpoints [m]

# Scatterer geometry
x_scatterer  = 2.000                          # fixed lateral position [m]
y_surface    = 0.9                            # air-ice interface [m from domain bottom]
y_baseline   = 0.224                          # baseline gprMax y of cylinder [m]
z_scatterer  = y_surface - y_baseline         # depth below surface [m] ≈ 0.676
radius_scat  = wavelength / 40               # cylinder radius ≈ 0.0028 m
z_top        = z_scatterer - radius_scat      # depth to cylinder top [m]

# Vertical shifts (downward = decreasing gprMax y = increasing depth)
shifts_lambda      = [1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
y_scatterers_shift = [round(y_baseline - s * wavelength, 3) for s in shifts_lambda]
z_depths_shift     = [round(y_surface - y, 3) for y in y_scatterers_shift]

labels     = ['1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']
labels_all = ['Baseline'] + labels

# Migration depth grid
z_img = np.linspace(0.0, 0.9, 180)   # full ice thickness (0.9 m), same ~5 mm sample spacing

# Study root
STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\vertical_timelapse_study')

print(f'λ_ice        = {wavelength*1e3:.1f} mm')
print(f'v_ice        = {v_ice:.5f} m/ns,  v_mig = {v_mig:.5f} m/ns')
print(f't0_ns        = {t0_ns:.3f} ns')
print(f'z_scatterer  = {z_scatterer:.3f} m  (baseline centre),  z_top = {z_top:.4f} m')
print(f'x_scatterer  = {x_scatterer:.3f} m  (fixed)')
print(f'x_traces     : {x_traces[0]:.3f} → {x_traces[-1]:.3f} m  ({n_traces} traces)')
print()
print('Scatterer y-positions (gprMax) and depths:')
print(f'  Baseline : y = {y_baseline:.3f} m   depth = {z_scatterer:.3f} m')
for lbl, y, z in zip(labels, y_scatterers_shift, z_depths_shift):
    print(f'  {lbl:>8} : y = {y:.3f} m   depth = {z:.3f} m')


# Importing Noise

In [ ]:
import pickle
from scipy import stats
with open('laplace_noise_model.pkl', 'rb') as f:
    noise_model = pickle.load(f)

with open('laplace_noise_model_pre_gain.pkl', 'rb') as f:
    noise_model_pre_gain = pickle.load(f)


## Calculate Geometry Factors

In [ ]:
# CFL check
dx_required = (v_ice / 4.5) / 10  # based on 4 GHz (highest Ricker component) / 10
if 0.001 < dx_required:
    print('Discretisation is sufficiently small')
if radius_scat < dx_required:
    print('Scatterer radius is smaller than discretisation')

# gprMax y-values for each scenario (baseline + 6 shifts)
y_all = [y_baseline] + y_scatterers_shift


# Load Cached Results (optional shortcut)

In [ ]:
USE_CACHED_RESULTS = True

if USE_CACHED_RESULTS:
    _static    = np.load(STUDY_ROOT / 'vertical_static_results.npz')
    _static_n  = np.load(STUDY_ROOT / 'vertical_static_results_noisy.npz')
    _mig       = np.load(STUDY_ROOT / 'vertical_migrated_results.npz')
    _mig_diff  = np.load(STUDY_ROOT / 'vertical_difference_migrated_results.npz')
    _mig_n     = np.load(STUDY_ROOT / 'vertical_migrated_results_noisy.npz')

    # ── Raw (background-subtracted) B-scans ─────────────────────────────────
    data_static    = list(zip(labels_all, _static['data_static']))
    data_noisy     = list(zip(labels_all, _static_n['data_static']))
    outputs_static = [d for _, d in data_static]
    outputs_noisy  = [d for _, d in data_noisy]

    time_ns     = _static['time_ns']
    dt          = float(_static['dt'])
    dt_ns       = dt * 1e9
    n_t         = outputs_static[0].shape[0]
    NOISE_LEVEL = float(_static_n['noise_level'])

    # z_depths_all normally built inside the Kirchhoff migration cell;
    # index 0 = Baseline depth, indices 1-6 = the six shifted depths.
    z_depths_all = [z_scatterer] + z_depths_shift
    t_apex       = [2 * z / v_ice for z in z_depths_all]   # expected apex arrival time per scenario

    # ── Migrated images (migration grid: n_z x n_x), clean ──────────────────
    migrated    = dict(zip(_mig['scenarios'], _mig['kirchhoff']))
    migrated_gz = dict(zip(_mig['scenarios'], _mig['gazdag']))
    migrated_bp = dict(zip(_mig['scenarios'], _mig['backprop']))   # already reprojected onto the migration grid

    # ── Migrated images, noisy ───────────────────────────────────────────────
    migrated_noisy    = dict(zip(_mig_n['scenarios'], _mig_n['kirchhoff']))
    migrated_gz_noisy = dict(zip(_mig_n['scenarios'], _mig_n['gazdag']))
    migrated_bp_noisy = dict(zip(_mig_n['scenarios'], _mig_n['backprop']))

    # ── TimeLapse differences, clean (cached directly) ───────────────────────
    migrated_diff    = dict(zip(_mig_diff['scenarios'], _mig_diff['kirchhoff_diff']))
    migrated_gz_diff = dict(zip(_mig_diff['scenarios'], _mig_diff['gazdag_diff']))
    migrated_bp_diff = dict(zip(_mig_diff['scenarios'], _mig_diff['backprop_diff']))   # migration-grid reprojection
    methods          = list(_mig_diff['methods'])

    # ── TimeLapse differences, noisy — NOT separately cached (no
    # vertical_difference_migrated_results_noisy.npz is ever saved), but the
    # subtraction is a cheap in-memory op on the cached full-scenario noisy
    # dicts, exactly mirroring the notebook's own "Migration (Noisy)" diff
    # cells, so it's reconstructed here rather than treated as uncovered.
    migrated_diff_noisy    = {lbl: migrated_noisy[lbl]    - migrated_noisy['Baseline']    for lbl in labels}
    migrated_gz_diff_noisy = {lbl: migrated_gz_noisy[lbl] - migrated_gz_noisy['Baseline'] for lbl in labels}

    extent_mig   = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
    extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]

    ANGLE_AP = 40   # must match the migration cell's value; only used for title text here

    # ── Analysis section (method-comparison grid + vertical PSF grid) ───────
    # Unusually (compared to sibling notebooks), the Analysis section here
    # only ever consumes migration-grid-reprojected images, and that
    # reprojected form (imgs[i][2] in the original cell) is exactly what's
    # cached as 'backprop_diff' in the difference archive — so this section
    # IS fully reconstructable from cache, unlike the raw wavefield plots
    # below.
    n_m, n_s = len(methods), len(labels)
    imgs = [[None] * n_m for _ in range(n_s)]
    for i, lbl in enumerate(labels):
        imgs[i][0] = migrated_diff.get(lbl)
        imgs[i][1] = migrated_gz_diff.get(lbl)
        imgs[i][2] = migrated_bp_diff.get(lbl)

    print(f'Loaded cached results from {STUDY_ROOT}')
    print(f'  data_static / data_noisy                         : {len(data_static)} scenarios, shape {outputs_static[0].shape}')
    print(f'  migrated / migrated_gz / migrated_bp              : {len(migrated)} scenarios, shape {next(iter(migrated.values())).shape}')
    print(f'  migrated_diff / migrated_gz_diff / migrated_bp_diff : {len(migrated_diff)} scenarios (no Baseline)')
    print(f'  migrated_noisy / migrated_gz_noisy / migrated_bp_noisy : {len(migrated_noisy)} scenarios')
    print(f'  migrated_diff_noisy / migrated_gz_diff_noisy      : reconstructed in-memory (not on disk), {len(migrated_diff_noisy)} scenarios')
    print(f'  imgs (Analysis grid, {n_s} x {n_m})                : ready for plot_method_comparison_grid / plot_psf_grid')
    print(f'  NOISE_LEVEL = {NOISE_LEVEL}')
    print()
    print('NOT covered by this shortcut (re-run the real cells if you need these):')
    print("  * 'Loading in the data...' section's `data_pre` / `output_background` /")
    print('    `raw_outputs` (the pre-background-subtraction traces, incl. the separate')
    print("    Background panel) - only the background-subtracted data_static/data_noisy")
    print('    are cached, so the "Background, Baseline and Vertical TimeLapsed Models"')
    print('    B-scan grid (first cell of "Visualisation") cannot be reconstructed.')
    print("  * Back-Propagation section's raw x-y-domain wavefield snapshots")
    print('    (`focus_frames`, `focus_frames_noisy`, `bp_ez_diff`, `mag_frames`,')
    print('    `ez_frames`) - these come from reading .vti snapshot files written by')
    print('    gprMax and are never saved to .npz. Only their migration-grid')
    print('    reprojection is cached (migrated_bp / migrated_bp_noisy / migrated_bp_diff,')
    print('    extent_mig domain), which is enough for the Analysis section above but NOT')
    print('    for the raw full-domain (extent_full=[0,4,0,1]) plot_wavefield_grid plots')
    print('    in the "Back-Propagation Migration" sections (clean and noisy).')

## Background model (no scatterers)

In [ ]:
%%writefile vertical_timelapse_study/background/background.in

#title: GPR Vertical TimeLapse Study - background
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 background n
#messages: y


## Baseline (static scatterer at x = 2.0 m, y = 0.224 m)

In [ ]:
%%writefile vertical_timelapse_study/baseline/baseline.in

#title: GPR Vertical TimeLapse Study - baseline
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.224 0 2.000 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 baseline n
#messages: y


## 1λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_1lambda/shift_1lambda.in

#title: GPR Vertical TimeLapse Study - 1λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.111 0 2.000 0.111 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_1lambda n
#messages: y


## ½λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_0p5lambda/shift_0p5lambda.in

#title: GPR Vertical TimeLapse Study - ½λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.168 0 2.000 0.168 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_0p5lambda n
#messages: y


## ¼λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_0p25lambda/shift_0p25lambda.in

#title: GPR Vertical TimeLapse Study - ¼λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.196 0 2.000 0.196 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_0p25lambda n
#messages: y


## ⅛λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_0p125lambda/shift_0p125lambda.in

#title: GPR Vertical TimeLapse Study - 1/8λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.210 0 2.000 0.210 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_0p125lambda n
#messages: y


## ¹⁄₁₆λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_0p0625lambda/shift_0p0625lambda.in

#title: GPR Vertical TimeLapse Study - 1/16λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.217 0 2.000 0.217 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_0p0625lambda n
#messages: y


## ¹⁄₃₂λ vertical shift

In [ ]:
%%writefile vertical_timelapse_study/shift_0p03125lambda/shift_0p03125lambda.in

#title: GPR Vertical TimeLapse Study - 1/32λ vertical shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Single point scatterer (PEC)
#cylinder: 2.000 0.220 0 2.000 0.220 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 shift_0p03125lambda n
#messages: y


# Visualising Model Geometry

In [ ]:
from matplotlib.patches import Rectangle, Circle
import matplotlib.pyplot as plt

# ── Geometry constants ────────────────────────────────────────────────────────
domain_y   = 1.0
dx_grid    = 0.001
pml_t      = 10 * dx_grid   # = 0.010 m
air_h      = domain_y - y_surface   # = 0.1 m
rx_off     = 0.1

# Depth coordinate (0 = ice surface, positive = deeper)
d_top  = -air_h           # -0.1 m
d_scat = z_scatterer      # 0.676 m  (baseline cylinder centre depth)

# All y positions and depth labels
_y_all  = [y_baseline] + y_scatterers_shift
_z_all  = [z_scatterer] + z_depths_shift
_labels = ['Baseline'] + labels
cmap_pts = plt.cm.plasma

# ── Figure 1: Full domain cross-section ──────────────────────────────────────
x_src_arr = 0.100 + np.arange(n_traces) * trace_step
fig1, ax1 = plot_domain_geometry(
    domain_x, domain_y, y_surface, pml_t,
    x_src=x_src_arr, tx_stride=5, rx_offset=rx_off, eps_r=eps_r,
)

# Baseline scatterer (radius x4 for visibility)
r_vis = 4 * radius_scat
ax1.add_patch(Circle((x_scatterer, d_scat), r_vis,
                      facecolor='tomato', edgecolor='#800', lw=0.8, zorder=5,
                      label='scatterer (baseline)'))

ax1.set_xlim(0, domain_x)
ax1.set_ylim(y_surface, d_top)
ax1.set_xlabel('x [m]', fontsize=11)
ax1.set_ylabel('Depth [m]', fontsize=11)
ax1.set_title(
    f'Vertical TimeLapse Study — Model Geometry  '
    f'(domain {domain_x:.1f}×{domain_y:.1f} m, Δx = {dx_grid*1e3:.0f} mm, '
    f'PML = {pml_t*1e3:.0f} mm, f_c = {f_c_GHz} GHz, λ = {wavelength*1e3:.1f} mm)  '
    f'tomato = baseline scatterer at x = {x_scatterer:.1f} m',
    fontsize=10
)
ax1.set_aspect('equal', adjustable='box')
ax1.legend(loc='upper right', fontsize=9, ncol=2)
ax1.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Figure 2: Zoomed – all scatterer depths at x = 2.0 m ─────────────────────
z_min = min(_z_all); z_max = max(_z_all)
z_cen = (z_min + z_max) / 2
z_hw  = max((z_max - z_min) / 2 + 5 * radius_scat, 8 * radius_scat)
x_hw  = 0.08  # half-width of the x-window around x_scatterer [m]

fig2, ax2 = plt.subplots(figsize=(8, 10))

ax2.add_patch(Rectangle((x_scatterer - x_hw, z_cen - z_hw - 0.01),
                         2 * x_hw, 2 * z_hw + 0.04,
                         facecolor='#cce5ff', edgecolor='#aac', lw=0.5))

for i, (lbl, z_d) in enumerate(zip(_labels, _z_all)):
    col = cmap_pts(i / max(len(_labels) - 1, 1))
    ax2.add_patch(Circle((x_scatterer, z_d), radius_scat,
                          facecolor=col, edgecolor='black', lw=0.6, zorder=5))
    ax2.text(x_scatterer + x_hw * 0.5, z_d, f'  {lbl}',
             ha='left', va='center', fontsize=10)

# Arrow showing max shift
ax2.annotate('', xy=(x_scatterer, _z_all[1]),
             xytext=(x_scatterer, _z_all[0]),
             arrowprops=dict(arrowstyle='->', color='k', lw=1.0))
ax2.text(x_scatterer - x_hw * 0.4, (_z_all[0] + _z_all[1]) / 2, f'max shift {(_z_all[1]-_z_all[0])*1e3:.1f} mm = 1λ', ha='right', va='center', fontsize=9)

ax2.set_xlim(x_scatterer - x_hw, x_scatterer + x_hw)
ax2.set_ylim(z_cen + z_hw + 0.02, z_cen - z_hw - 0.01)
ax2.set_xlabel('x [m]', fontsize=11)
ax2.set_ylabel('Depth below surface [m]', fontsize=11)
ax2.set_title(
    f'Moving scatterer — all 7 scenarios  ' f'(r = {radius_scat*1e3:.1f} mm, x = {x_scatterer:.1f} m)'
    f'Baseline depth = {z_scatterer:.3f} m,  max shift = 1λ downward',
    fontsize=11
)
ax2.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.show()


# Generating Geometry Files and Merged B-scans
Only uncomment and run when necessary.

In [ ]:
# Uncomment and run to execute gprMax simulations for each scenario.
# Each scenario requires n_traces = 380 model runs.

# from gprMax.gprMax import api
# api(r'vertical_timelapse_study/background/background.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/baseline/baseline.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_1lambda/shift_1lambda.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_0p5lambda/shift_0p5lambda.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_0p25lambda/shift_0p25lambda.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_0p125lambda/shift_0p125lambda.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_0p0625lambda/shift_0p0625lambda.in', n=380, geometry_fixed=True)
# api(r'vertical_timelapse_study/shift_0p03125lambda/shift_0p03125lambda.in', n=380, geometry_fixed=True)


In [ ]:
# Uncomment and run to merge per-trace .out files into a single merged .out file.

# from tools.outputfiles_merge import merge_files
# merge_files(r'vertical_timelapse_study/background/background', True)
# merge_files(r'vertical_timelapse_study/baseline/baseline', True)
# merge_files(r'vertical_timelapse_study/shift_1lambda/shift_1lambda', True)
# merge_files(r'vertical_timelapse_study/shift_0p5lambda/shift_0p5lambda', True)
# merge_files(r'vertical_timelapse_study/shift_0p25lambda/shift_0p25lambda', True)
# merge_files(r'vertical_timelapse_study/shift_0p125lambda/shift_0p125lambda', True)
# merge_files(r'vertical_timelapse_study/shift_0p0625lambda/shift_0p0625lambda', True)
# merge_files(r'vertical_timelapse_study/shift_0p03125lambda/shift_0p03125lambda', True)


# Loading in the data and removing direct wave

In [ ]:
rxnumber    = 1
rxcomponent = 'Ez'

output_background, dt = get_output_data(
    'vertical_timelapse_study/background/background_merged.out', rxnumber, rxcomponent
)

_paths = [
    'vertical_timelapse_study/baseline/baseline_merged.out',
    'vertical_timelapse_study/shift_1lambda/shift_1lambda_merged.out',
    'vertical_timelapse_study/shift_0p5lambda/shift_0p5lambda_merged.out',
    'vertical_timelapse_study/shift_0p25lambda/shift_0p25lambda_merged.out',
    'vertical_timelapse_study/shift_0p125lambda/shift_0p125lambda_merged.out',
    'vertical_timelapse_study/shift_0p0625lambda/shift_0p0625lambda_merged.out',
    'vertical_timelapse_study/shift_0p03125lambda/shift_0p03125lambda_merged.out',
]
raw_outputs    = [get_output_data(p, rxnumber, rxcomponent)[0] for p in _paths]
outputs_static = [r - output_background for r in raw_outputs]   # background-subtracted

dt_ns   = dt * 1e9
n_t     = outputs_static[0].shape[0]
time_ns = np.arange(n_t) * dt_ns

data_pre    = [('Background', output_background)] + list(zip(labels_all, raw_outputs))
data_static = list(zip(labels_all, outputs_static))

print(f'dt = {dt_ns:.6f} ns,  n_t = {n_t},  t_max = {time_ns[-1]:.2f} ns')
print(f'Data shape (n_t × n_tr): {outputs_static[0].shape}')

# Expected apex arrival times for each scenario (2 * depth / v_ice)
t_apex = [2 * z / v_ice for z in [z_scatterer] + z_depths_shift]
print('\nExpected apex arrival times:')
for lbl, t in zip(labels_all, t_apex):
    print(f'  {lbl:>8}: {t:.3f} ns')


# Visualisation

In [ ]:
extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]

def _raw_apex_marker(ax, i):
    if i > 0:   # skip the 'Background' panel (index 0)
        ax.axhline(t_apex[i - 1], color='green', linestyle='--', linewidth=1)

fig, axes = plot_bscan_grid(
    data_pre, x_traces, time_ns, figsize=(26, 5),
    title='GPR B-Scans — Background, Baseline and Vertical TimeLapsed Models',
    markers=[_raw_apex_marker] * len(data_pre),
)
plt.show()


In [ ]:
def _static_apex_marker(ax, i):
    ax.axhline(t_apex[i], color='green', linestyle='--', linewidth=1, label=f't = {t_apex[i]:.2f} ns')
    ax.axvline(x_scatterer, color='green', linestyle=':', linewidth=0.8)
    ax.legend(fontsize=7, loc='upper right')

fig, axes = plot_bscan_grid(
    data_static, x_traces, time_ns, figsize=(24, 5),
    title='GPR B-Scans — Background Subtracted  (green dashed = expected apex time)',
    markers=[_static_apex_marker] * len(data_static),
)
plt.show()


Create Noisy Data

In [ ]:
# ── Apply the fitted Laplace noise model to every background-subtracted B-scan ──
# The Laplace fit (loc/scale) was estimated from the real, fully-processed data
# pipeline ('Spherical Gain' stage), whose amplitudes are ~4 orders of magnitude
# larger than these synthetic Ez B-scans. We keep the Laplace *shape* (loc=0,
# heavier tails than Gaussian) but rescale it so its std is 10% of each B-scan's
# own signal std — a light, realistic noise level rather than the raw fitted scale.
NOISE_LEVEL = 0.1   # target noise std as a fraction of each B-scan's signal std
rng = np.random.default_rng(0)

outputs_noisy = []
for raw in outputs_static:
    target_std    = NOISE_LEVEL * raw.std()
    scaled_scale  = target_std / np.sqrt(2)   # Var(Laplace) = 2 * scale**2
    synthetic_noise = stats.laplace.rvs(
        loc=noise_model_pre_gain['loc'], scale=scaled_scale,
        size=raw.shape, random_state=rng,
    )
    outputs_noisy.append(raw + synthetic_noise)

data_noisy = list(zip(labels_all, outputs_noisy))

print(f"Applied Laplace noise (shape from '{noise_model_pre_gain['stage']}' fit, "
      f"rescaled to {NOISE_LEVEL:.0%} of signal std) to {len(outputs_noisy)} B-scans")


In [ ]:
def _noisy_apex_marker(ax, i):
    ax.axhline(t_apex[i], color='green', linestyle='--', linewidth=1, label=f't = {t_apex[i]:.2f} ns')
    ax.axvline(x_scatterer, color='green', linestyle=':', linewidth=0.8)
    ax.legend(fontsize=7, loc='upper right')

fig, axes = plot_bscan_grid(
    data_noisy, x_traces, time_ns, figsize=(24, 5),
    title='GPR B-Scans — With Synthetic Laplace Noise (10% of signal std)',
    markers=[_noisy_apex_marker] * len(data_noisy),
)
plt.show()


In [ ]:
# -- Frequency spectra: clean vs noisy B-scans -----------------------------------
# Row 1: amplitude spectrum (mean across traces) of each clean (background-
# subtracted) B-scan in data_static. Row 2: the same for the noisy B-scans in
# data_noisy -- compare rows to see how much broadband content the Laplace
# noise injection adds on top of the underlying Ricker wavelet bandwidth.
# norm='forward' makes amplitudes comparable across scenarios/notebooks (this
# cell previously omitted it -- confirmed bug fix, see sibling notebook).
freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

spectra_clean = [(f'{label} (clean)',
                   np.abs(np.fft.rfft(d, axis=0, norm='forward')).mean(axis=1))
                  for label, d in data_static]
spectra_noisy = [(f'{label} (noisy)',
                   np.abs(np.fft.rfft(d, axis=0, norm='forward')).mean(axis=1))
                  for label, d in data_noisy]

fig, axes = plot_spectrum_grid(
    [spectra_clean, spectra_noisy], freqs_ghz, figsize=(22, 8),
    title=f'B-scan Frequency Spectra -- Clean vs Noisy  |  f_c={f_c_GHz} GHz',
)
plt.show()


# Saving the Data

In [ ]:
# ── Save non-migrated (raw, background-subtracted) B-scan data ────────────
# Stacks all 7 scenarios (Baseline + 6 shifts) from data_static into a single
# array and saves to vertical_static_results.npz.
# Index 0 = Baseline, indices 1-6 = shift scenarios matching labels_all[1:].

static_all = np.stack([d.astype(float) for _, d in data_static], axis=0)  # (7, n_t, n_traces)

save_path_static = STUDY_ROOT / "vertical_static_results.npz"

np.savez_compressed(
    save_path_static,
    data_static  = static_all,                                  # (7, n_t, n_traces)
    scenarios    = np.array(labels_all, dtype="U20"),            # (7,)  Baseline + 6 shifts
    shift_lambda = np.array([0] + shifts_lambda),                # (7,)  0 = baseline
    x_traces     = x_traces,
    time_ns      = time_ns,
    dt           = np.float64(dt),
    x_scatterer  = np.float64(x_scatterer),
    y_baseline   = np.float64(y_baseline),
    y_scatterers = np.array([y_baseline] + y_scatterers_shift),
    z_scatterer  = np.float64(z_scatterer),
    z_depths     = np.array([z_scatterer] + z_depths_shift),
    z_top        = np.float64(z_top),
)

print(f"Saved -> {save_path_static}")
print(f"\nStacked array  (n_scenarios={static_all.shape[0]}, n_t={static_all.shape[1]}, n_tr={static_all.shape[2]}):")
print(f"  data_static   {static_all.shape}")
print(f"\nUsage example:")
print(f"  d = np.load(str(STUDY_ROOT / 'vertical_static_results.npz'), allow_pickle=False)")
print(f"  d['data_static'][0]   # Baseline B-scan -> shape (n_t, n_tr)")
print(f"  d['data_static'][3]   # 1/4 lambda shift B-scan -> shape (n_t, n_tr)")


# Kirchhoff Migration

Delay-and-sum migration via the **PyLops zero-offset Kirchhoff operator**.

## Tapering & t0 Shift

Two pre-processing steps applied to every B-scan before migration:

| Step | Purpose |
|---|---|
| End taper (half-cosine) | Suppress ringing from late-time hyperbola tails |
| Exponential decay taper | Down-weight late, noisy arrivals |
| t0 roll | Compensate for Ricker wavelet peak delay so zero-offset apex aligns with t=0 |

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))
from helper_functions.migration import (PylopsKirchoffMigration, gazdag_migration,
                                        write_backprop_files, dispersion_limited_cutoff,
                                        lowpass_filter_excitation)

In [ ]:
# ── Taper & t0-shift parameters ────────────────────────────────────────────
taper_end_ns   = 14.0   # cosine ramp onset [ns]
taper_decay_ns = 1.0    # exponential decay time constant [ns]
ANGLE_AP       = 40     # Kirchhoff aperture half-angle [degrees]

# ── Functions ───────────────────────────────────────────────────────────────
def make_end_taper(n_samples, taper_samples):
    win = np.ones(n_samples)
    if taper_samples > 0:
        ramp = 0.5 * (1 + np.cos(np.pi * np.arange(taper_samples) / taper_samples))
        win[-taper_samples:] = ramp
    return win

def make_decay_taper(time_ns, tau_ns):
    return np.exp(-time_ns / tau_ns)

def preprocess(data_nt_ntr, dt_ns, t0_ns, time_ns):
    bscan   = data_nt_ntr.T.copy()
    n_t     = bscan.shape[1]
    w_end   = make_end_taper(n_t, int(round(taper_end_ns / dt_ns)))
    w_decay = make_decay_taper(time_ns, taper_decay_ns)
    tapered = bscan * (w_end * w_decay)[np.newaxis, :]
    t0_samp = int(round(t0_ns / dt_ns))
    shifted = np.roll(tapered, -t0_samp, axis=1)
    shifted[:, -t0_samp:] = 0.0
    return tapered, shifted

# ── Visualise effect on the 1λ dataset ────────────────────────────────────
_demo = outputs_static[1]    # (n_t, n_tr) - 1λ background-subtracted
_raw_tr, _shifted = preprocess(_demo, dt_ns, t0_ns, time_ns)
_tapered = _raw_tr

w_end_vis   = make_end_taper(n_t, int(round(taper_end_ns   / dt_ns)))
w_decay_vis = make_decay_taper(time_ns, taper_decay_ns)
taper_combined = w_end_vis * w_decay_vis

mid_tr = n_traces // 2

fig, axes = plot_trace_comparison(
    traces=[
        [('Raw', _demo[:, mid_tr], 'b'), ('Tapered', _tapered[mid_tr], 'r')],
        [('Tapered', _tapered[mid_tr], 'r'), (f't0-shifted (−{t0_ns:.3f} ns)', _shifted[mid_tr], 'g')],
        [],
    ],
    time_ns=time_ns,
    title='Effect of Tapering and t0 Shift — 1λ dataset, single trace',
    taper_curve=taper_combined,
    vline=taper_end_ns,
    figsize=(16, 4),
)
axes[0].set_ylabel('Ez [V/m]', fontsize=11)
axes[0].set_title(f'Trace #{mid_tr} — raw vs tapered')
axes[1].set_ylabel('Ez [V/m]', fontsize=11)
axes[1].set_title(f'Trace #{mid_tr} — tapered vs t0-shifted')
axes[2].set_title(f'Combined taper  (decay τ={taper_decay_ns} ns, end from {taper_end_ns} ns)')
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

# ── B-scan effect of tapering and t0 shift ──────────────────────────────────
fig, axes = plot_bscan_grid(
    data=[
        ('Raw (background subtracted)', _demo),
        (f'Tapered  (decay τ={taper_decay_ns} ns, end from {taper_end_ns} ns)', _tapered.T),
        (f't0-shifted (−{t0_ns:.3f} ns)', _shifted.T),
    ],
    x_traces=x_traces, time_ns=time_ns, shared_colorbar=False,
    vmax_headroom=0.5, figsize=(18, 5),
    title='B-scan effect of tapering and t0 shift — 1λ dataset',
)
plt.show()


In [ ]:
# ── Kirchhoff migration on all 7 datasets ─────────────────────────────────
migrated = {}
z_depths_all = [z_scatterer] + z_depths_shift   # depth of scatterer per scenario

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)
    print(f'[{label}] Running Kirchhoff ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated[label] = img


In [ ]:
# ── Plot: full extent ──────────────────────────────────────────────────────
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

def _mark_baseline_current(axes, y_baseline_pos, y_current_list, x=x_scatterer, legend=None):
    """Overlay the fixed baseline-position star + a per-panel current-position
    triangle on each axis of a migrated/back-propagation grid.

    In this vertical-shift study the baseline and current markers differ in
    *depth* (y), not in x, for every panel -- the opposite of the lateral-shift
    studies that plot_migrated_grid/plot_wavefield_grid's built-in
    marker_x_baseline was designed for (it shares the current marker's y for
    the baseline star, only varying x). That mechanism can't express "same x,
    different depth", so the baseline+current pair is added here instead,
    directly on the axes the shared grid function returns.

    legend: None draws no legend, 'first' legends only axes.ravel()[0],
    'all' legends every used panel (mirrors the three conventions used
    across this notebook's original migrated/back-propagation grid cells).
    """
    flat = np.atleast_1d(axes).ravel()
    for ax, y_cur in zip(flat, y_current_list):
        ax.plot(x, y_baseline_pos, 'g*', ms=10, zorder=5, label='baseline depth')
        ax.plot(x, y_cur, 'g^', ms=10, zorder=5, label='current depth')
    if legend == 'all':
        for ax in flat[:len(y_current_list)]:
            ax.legend(fontsize=8, loc='upper right')
    elif legend == 'first':
        flat[0].legend(fontsize=8, loc='upper right')

fig, axes = plot_migrated_grid(
    migrated, extent_mig, ncols=4,
    title=f'Kirchhoff Migration — All 7 Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all)
plt.show()

# ── Plot: zoomed on scatterer region ────────────────────────────────────────
fig, axes = plot_migrated_grid(
    migrated, extent_mig, ncols=4,
    title=f'Kirchhoff Migration (zoomed)  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all, legend='all')
plt.show()


In [ ]:
# ── Kirchhoff timelapse differences (migrated − migrated_baseline) ─────────
migrated_diff = {label: migrated[label] - migrated['Baseline'] for label in labels}

fig, axes = plot_migrated_grid(
    migrated_diff, extent_mig, ncols=3,
    title=f'Kirchhoff Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()

# ── Zoomed ─────────────────────────────────────────────────────────────────
fig, axes = plot_migrated_grid(
    migrated_diff, extent_mig, ncols=3,
    title='Kirchhoff Migration — TimeLapse Differences (zoomed)',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_diff)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()


# Gazdag Phase-Shift Migration

f-k domain migration via **PyLops `PhaseShift`** downward continuation.

In [ ]:
import pylops

## Run Migration on All Datasets

In [ ]:
# ── Gazdag phase-shift migration on all 7 datasets ─────────────────────────
migrated_gz = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)
    print(f'[{label}] Running Gazdag ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) – time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz[label] = img


In [ ]:
# ── Plot: full extent ──────────────────────────────────────────────────────
fig, axes = plot_migrated_grid(
    migrated_gz, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration — All 7 Datasets  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all)
plt.show()

# ── Plot: zoomed ─────────────────────────────────────────────────────────
fig, axes = plot_migrated_grid(
    migrated_gz, extent_mig, ncols=4,
    title='Gazdag Phase-Shift Migration (zoomed)  |  f_c={f_c_GHz} GHz',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_gz)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all, legend='all')
plt.show()


In [ ]:
# ── Gazdag timelapse differences ──────────────────────────────────────────
migrated_gz_diff = {label: migrated_gz[label] - migrated_gz['Baseline'] for label in labels}

fig, axes = plot_migrated_grid(
    migrated_gz_diff, extent_mig, ncols=3,
    title=f'Gazdag Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()

# ── Zoomed ─────────────────────────────────────────────────────────────────
fig, axes = plot_migrated_grid(
    migrated_gz_diff, extent_mig, ncols=3,
    title='Gazdag Migration — TimeLapse Differences (zoomed)',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_gz_diff)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()


# Back-Propagation Migration

Time-reversal migration: re-inject the time-reversed, normalised B-scan as sources at the receiver positions and let gprMax propagate the wavefield backward in time.

In [ ]:
# -- Back-propagation parameters ------------------------------------------
slugs_all = ['baseline', '1lambda', '0p5lambda', '0p25lambda', '0p125lambda', '0p0625lambda', '0p03125lambda']
STRIDE   = 1    # use every Nth trace
N_SNAP   = 30   # number of snapshots in the focus window
SNAP_WIN = 1.0  # [ns] window before focus time

# Some lambda fractions inject sub-wavelength-spaced sources whose time-
# reversed wavefield carries spectral content above gprMax's numerical-
# dispersion limit (cells/wavelength), causing a 'Non-physical wave
# propagation' error. Low-pass filter each excitation file right after
# writing it so every .in file is runnable. eps_r*4 / dx=0.001 mirror the
# half-velocity material and grid hardcoded in write_backprop_files() --
# see helper_functions/migration.py.
#
# Filtering alone isn't always enough: the outermost virtual sources sit at
# the edge of the migration aperture and can carry near-Nyquist content
# baked into the raw time-reversed data (seen for 0p5lambda: a single-
# sample swing of ~0.95 out of +-1). No low-pass filter can safely remove
# that without either leaving it untouched or, if pushed too aggressively,
# causing ringing that makes the measured frequency even worse. EDGE_DROP
# zeros those outermost sources entirely instead -- standard aperture-
# limiting practice, verified against gprMax for all 7 datasets here.
EDGE_DROP = 3
cutoff_hz = dispersion_limited_cutoff(eps_r=4.0 * eps_r, dx=0.001)

# -- Generate files for all 7 datasets --------------------------------------
print(f'Back-propagation file generation  (stride={STRIDE}, {N_SNAP} snapshots)' + chr(10))
bp_paths = {}
for label, slug, raw in zip(labels_all, slugs_all, outputs_static):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, label, slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN
    )
    lowpass_filter_excitation(in_path.parent / 'excitation.txt', cutoff_hz, edge_exclude=EDGE_DROP)
    bp_paths[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB  |  '
          f'low-pass={cutoff_hz/1e9:.1f} GHz  |  edge_drop={EDGE_DROP}')
    print(f'           {in_path}')

In [ ]:
import pyvista

# ── Load focus frame for all 7 datasets ───────────────────────────────────
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_all)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} – skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4,
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames[label]["t_actual"]:.3f} ns')

extent_full = [0, 4.0, 0, 1]
margin_x    = 0.4
margin_y    = 0.10
# x-zoom: centred on scatterer
x_zoom_lo = x_scatterer - margin_x
x_zoom_hi = x_scatterer + margin_x
# y-zoom: cover baseline y=0.224 down to 1λ y=0.111, with margin
y_zoom_lo = max(0.0, y_scatterers_shift[0] - margin_y)   # deepest shifted y - margin
y_zoom_hi = y_baseline + margin_y                          # baseline y + margin

# y_all (gprMax y per scenario) was already set in the CFL-check cell above;
# frame['i'] (each frame's true index into labels_all/y_all) is used -- not
# positional loop order -- so markers stay correct even if some scenarios'
# snapshots are missing (focus_frames can be a strict subset of labels_all).
y_current_bp = [y_all[frame['i']] for frame in focus_frames.values()]
mag_frames = {f'{label}  |  t={frame["t_actual"]:.2f} ns': frame['mag']
              for label, frame in focus_frames.items()}
ez_frames  = {f'{label}  |  t={frame["t_actual"]:.2f} ns': frame['ez']
              for label, frame in focus_frames.items()}

# ── |E| magnitude – full extent ──────────────────────────────────────────
fig, axes = plot_wavefield_grid(
    mag_frames, extent_full, field='magnitude', ncols=4, y_surface=y_surface,
    title=f'Back-Propagation |E| — All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
)
_mark_baseline_current(axes, y_baseline, y_current_bp, legend='first')
plt.show()

# ── |E| magnitude – zoomed ─────────────────────────────────────────────
fig, axes = plot_wavefield_grid(
    mag_frames, extent_full, field='magnitude', ncols=4,
    title=f'Back-Propagation |E| (zoomed)  |  focus at {t_focus_ns:.2f} ns',
)
for ax in np.atleast_1d(axes).ravel()[:len(mag_frames)]:
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_zoom_lo, y_zoom_hi)
_mark_baseline_current(axes, y_baseline, y_current_bp, legend='all')
plt.show()

# ── Ez component – full extent ─────────────────────────────────────────
fig, axes = plot_wavefield_grid(
    ez_frames, extent_full, field='signed', ncols=4, y_surface=y_surface,
    title=f'Back-Propagation Ez — All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
)
_mark_baseline_current(axes, y_baseline, y_current_bp, legend='first')
plt.show()

# ── Ez component – zoomed ──────────────────────────────────────────────
fig, axes = plot_wavefield_grid(
    ez_frames, extent_full, field='signed', ncols=4,
    title=f'Back-Propagation Ez (zoomed)  |  focus at {t_focus_ns:.2f} ns',
)
for ax in np.atleast_1d(axes).ravel()[:len(ez_frames)]:
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_zoom_lo, y_zoom_hi)
_mark_baseline_current(axes, y_baseline, y_current_bp, legend='all')
plt.show()


In [ ]:
# ── Back-propagation timelapse differences (Ez − Ez_baseline) ────────────
bp_ez_diff = {label: focus_frames[label]['ez'] - focus_frames['Baseline']['ez']
              for label in labels if label in focus_frames}
y_current_diff = y_scatterers_shift[:len(bp_ez_diff)]

fig, axes = plot_wavefield_grid(
    bp_ez_diff, extent_full, field='signed', ncols=3, vmax_headroom=1.0,
    title=f'Back-Propagation — TimeLapse Differences Ez (Ez − Ez_baseline)  |  focus at {t_focus_ns:.2f} ns',
)
_mark_baseline_current(axes, y_baseline, y_current_diff, legend='all')
plt.show()

# ── Zoomed ──────────────────────────────────────────────────────────────────
fig, axes = plot_wavefield_grid(
    bp_ez_diff, extent_full, field='signed', ncols=3, vmax_headroom=1.0,
    title='Back-Propagation — TimeLapse Differences Ez (zoomed)',
)
for ax in np.atleast_1d(axes).ravel()[:len(bp_ez_diff)]:
    ax.set_xlim(x_zoom_lo, x_zoom_hi)
    ax.set_ylim(y_zoom_lo, y_zoom_hi)
_mark_baseline_current(axes, y_baseline, y_current_diff, legend='all')
plt.show()


# Analysis

In [ ]:
from scipy.signal import hilbert as scipy_hilbert
from scipy.interpolate import RegularGridInterpolator

# ── Collect timelapse difference images ────────────────────────────────────
methods = ['Kirchhoff', 'Gazdag', 'Back-prop']
n_m, n_s = len(methods), len(labels)
imgs = [[None] * n_m for _ in range(n_s)]

for i, lbl in enumerate(labels):
    try:    imgs[i][0] = migrated_diff[lbl]
    except (KeyError, NameError): pass
    try:    imgs[i][1] = migrated_gz_diff[lbl]
    except (KeyError, NameError): pass

# Back-propagation – reproject bp_ez_diff to migration grid
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

for i, lbl in enumerate(labels):
    if lbl not in bp_ez_diff:
        print(f'[{lbl}] bp_ez_diff missing – skipping')
        continue
    interp = RegularGridInterpolator(
        (y_ax_bp, x_ax_bp), bp_ez_diff[lbl],
        method='linear', bounds_error=False, fill_value=0.0
    )
    imgs[i][2] = interp((Yq, Xq))
    print(f'[{lbl}] back-prop diff reprojected  shape={imgs[i][2].shape}')

extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
xlim = (1.5, 2.5)
ylim = (0.84, 0.62)

fig, axes = plot_method_comparison_grid(
    imgs, extent_mig, methods, labels, envelope=False,
    marker_x=x_scatterer, marker_z=lambda i, j: z_depths_all[i + 1],
    xlim=xlim, ylim=ylim, vmax_percentile=100,
    title=f'Vertical TimeLapse Migration Comparison — Signed Amplitude  |  f_c={f_c_GHz} GHz',
)
for i in range(n_s):
    axes[i, 0].set_ylabel(labels[i], fontsize=11, fontweight='bold',
                           rotation=0, labelpad=36, va='center')
    for j in range(n_m):
        if imgs[i][j] is None:
            continue
        ax = axes[i, j]
        ax.plot(x_scatterer, z_depths_all[0], 'g*', ms=8, zorder=5, label='baseline')
        ax.axvline(x_scatterer, color='green', ls=':', lw=0.8, alpha=0.5)
        ax.tick_params(labelsize=7)
plt.show()


In [ ]:
# ── Normalised vertical PSF of timelapse difference at scatterer x-position ──
# Each panel shows the depth profile at x = x_scatterer (x = 2.0 m), with depth
# running vertically down the page (standard depth-profile convention).
# Blue  = signed amplitude (normalised to peak)
# Red   = Hilbert envelope
# Green dashed = shifted depth
# Green dotted  = baseline depth

ix_s = int(np.argmin(np.abs(x_traces - x_scatterer)))   # column index at x = 2.0 m
z_win = 3.0 * wavelength   # half-window in depth around scatterer centre [m]

profiles = [[None] * n_m for _ in range(n_s)]
for i, lbl in enumerate(labels):
    for j, method in enumerate(methods):
        img = imgs[i][j]
        if img is None:
            continue
        profile = img[:, ix_s].copy()
        peak = np.max(np.abs(profile))
        if peak > 0:
            profile /= peak
        profiles[i][j] = profile

fig, axes = plot_psf_grid(
    profiles, z_img, methods, labels, orientation='vertical',
    title='Normalised Vertical PSF — TimeLapse Difference at x = 2.0 m\n'
          'blue = amplitude  |  red = Hilbert envelope  '
          '|  green dashed = shifted depth  |  green dotted = baseline depth',
)
for i in range(n_s):
    axes[i, 0].set_ylabel('Depth z [m]', fontsize=8)
    for j in range(n_m):
        if profiles[i][j] is None:
            continue
        ax = axes[i, j]
        ax.axhline(z_depths_all[i + 1], color='green', ls='--', lw=1.2,
                   label=f'shifted depth {z_depths_all[i+1]:.3f} m')
        ax.axhline(z_depths_all[0], color='green', ls=':', lw=1.2,
                   label=f'baseline depth {z_depths_all[0]:.3f} m')
        ax.axvline(0, color='k', lw=0.4, alpha=0.4)
        z_centre = z_depths_all[i + 1]
        ax.set_ylim(z_centre + z_win, z_centre - z_win)
        ax.set_xlim(-1.15, 1.4)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.2)
    if i == n_s - 1:
        for j in range(n_m):
            axes[i, j].set_xlabel('Amplitude (normalised)', fontsize=8)

axes[0, 0].legend(fontsize=7, loc='upper right')
plt.show()


# Save Migrated Results as np arrays

In [ ]:
# ── Build structured numpy archive of all migrated difference results ──────
_ref_shape = imgs[0][0].shape   # (n_z, n_x)

def _safe_stack(method_idx):
    planes = []
    for i in range(n_s):
        arr = imgs[i][method_idx]
        planes.append(arr.astype(float) if arr is not None
                      else np.full(_ref_shape, np.nan))
    return np.stack(planes, axis=0)

diff_kirchhoff = _safe_stack(0)   # (6, n_z, n_x)
diff_gazdag    = _safe_stack(1)
diff_backprop  = _safe_stack(2)

save_path = STUDY_ROOT / 'vertical_difference_migrated_results.npz'

np.savez_compressed(
    save_path,
    kirchhoff_diff    = diff_kirchhoff,
    gazdag_diff       = diff_gazdag,
    backprop_diff     = diff_backprop,
    scenarios         = np.array(labels,  dtype='U20'),
    methods           = np.array(methods, dtype='U20'),
    shift_lambda      = np.array(shifts_lambda),
    y_scatterers      = np.array(y_scatterers_shift),
    z_depths          = np.array(z_depths_shift),
    x_traces          = x_traces,
    z_img             = z_img,
    x_scatterer       = np.float64(x_scatterer),
    y_baseline        = np.float64(y_baseline),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

print(f'Saved → {save_path}')
print(f'\nDifference arrays  (n_scenarios={n_s}, n_z={_ref_shape[0]}, n_x={_ref_shape[1]}):')
for name, arr in [('kirchhoff_diff', diff_kirchhoff),
                  ('gazdag_diff',    diff_gazdag),
                  ('backprop_diff',  diff_backprop)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f'  ({absent} scenario(s) absent → NaN)' if absent else ''
    print(f'  {name:<18}  {arr.shape}{note}')


In [ ]:
# ── Save normal (non-difference) migrated results from all methods ─────────
kirchhoff_all = np.stack([migrated[lbl].astype(float)    for lbl in labels_all], axis=0)  # (7, n_z, n_x)
gazdag_all    = np.stack([migrated_gz[lbl].astype(float) for lbl in labels_all], axis=0)

x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_planes = []
for lbl in labels_all:
    if lbl in focus_frames:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
        print(f'[{lbl}] focus_frames missing – filled with NaN')
backprop_all = np.stack(backprop_planes, axis=0)   # (7, n_z, n_x)

save_path_mig = STUDY_ROOT / 'vertical_migrated_results.npz'
np.savez_compressed(
    save_path_mig,
    kirchhoff         = kirchhoff_all,
    gazdag            = gazdag_all,
    backprop          = backprop_all,
    scenarios         = np.array(labels_all, dtype='U20'),
    shift_lambda      = np.array([0] + shifts_lambda),
    y_scatterers      = np.array([y_baseline] + y_scatterers_shift),
    z_depths          = np.array([z_scatterer] + z_depths_shift),
    x_traces          = x_traces,
    z_img             = z_img,
    x_scatterer       = np.float64(x_scatterer),
    y_baseline        = np.float64(y_baseline),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

print(f'Saved → {save_path_mig}')
print(f'\nStacked arrays  (n_scenarios=7, n_z={kirchhoff_all.shape[1]}, n_x={kirchhoff_all.shape[2]}):')
for name, arr in [('kirchhoff', kirchhoff_all), ('gazdag', gazdag_all), ('backprop', backprop_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f'  ({absent} absent → NaN)' if absent else ''
    print(f'  {name:<12}  {arr.shape}{note}')


# Application to Noisy Data

## Save Unmigrated Noisy B-scans

In [ ]:
# -- Save non-migrated (noisy, background-subtracted) B-scan data --------------
# Mirrors the "Saving the Data" step above, but for the Laplace-noise-augmented
# B-scans (data_noisy) created in the "Create Noisy Data" section.

noisy_static_all = np.stack([d.astype(float) for _, d in data_noisy], axis=0)  # (7, n_t, n_traces)

save_path_noisy_static = STUDY_ROOT / "vertical_static_results_noisy.npz"

np.savez_compressed(
    save_path_noisy_static,
    data_static  = noisy_static_all,                             # (7, n_t, n_traces)
    scenarios    = np.array(labels_all, dtype="U20"),             # (7,)  Baseline + 6 shifts
    shift_lambda = np.array([0] + shifts_lambda),                 # (7,)  0 = baseline
    x_traces     = x_traces,
    time_ns      = time_ns,
    dt           = np.float64(dt),
    x_scatterer  = np.float64(x_scatterer),
    y_baseline   = np.float64(y_baseline),
    y_scatterers = np.array([y_baseline] + y_scatterers_shift),
    z_scatterer  = np.float64(z_scatterer),
    z_depths     = np.array([z_scatterer] + z_depths_shift),
    z_top        = np.float64(z_top),
    noise_level  = np.float64(NOISE_LEVEL),
)

print(f"Saved -> {save_path_noisy_static}")
print(f"\nStacked array  (n_scenarios={noisy_static_all.shape[0]}, n_t={noisy_static_all.shape[1]}, n_tr={noisy_static_all.shape[2]}):")
print(f"  data_static   {noisy_static_all.shape}")


## Kirchhoff Migration (Noisy)

In [ ]:
# -- Kirchhoff migration on all 7 noisy datasets --------------------------------
migrated_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)
    print(f'[{label}] Running Kirchhoff (noisy) ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_noisy[label] = img


In [ ]:
# -- Plot: full extent ------------------------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_noisy, extent_mig, ncols=4,
    title=f'Kirchhoff Migration - All 7 Noisy Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP} deg',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all)
plt.show()

# -- Plot: zoomed on scatterer region ----------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_noisy, extent_mig, ncols=4,
    title=f'Kirchhoff Migration (Noisy, zoomed)  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP} deg',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_noisy)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all, legend='all')
plt.show()


In [ ]:
# -- Kirchhoff timelapse differences (noisy migrated - noisy migrated_baseline) --
migrated_diff_noisy = {label: migrated_noisy[label] - migrated_noisy['Baseline'] for label in labels}

fig, axes = plot_migrated_grid(
    migrated_diff_noisy, extent_mig, ncols=3,
    title=f'Kirchhoff Migration - Noisy TimeLapse Differences (migrated - migrated_baseline)  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()

# -- Zoomed -------------------------------------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_diff_noisy, extent_mig, ncols=3,
    title='Kirchhoff Migration - Noisy TimeLapse Differences (zoomed)',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_diff_noisy)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()


## Gazdag Phase-Shift Migration (Noisy)

In [ ]:
# -- Gazdag phase-shift migration on all 7 noisy datasets -----------------------
migrated_gz_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)
    print(f'[{label}] Running Gazdag (noisy) ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) - time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz_noisy[label] = img


In [ ]:
# -- Plot: full extent ------------------------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_gz_noisy, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration - All 7 Noisy Datasets  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all)
plt.show()

# -- Plot: zoomed -------------------------------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_gz_noisy, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration (Noisy, zoomed)  |  f_c={f_c_GHz} GHz',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_gz_noisy)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all, legend='all')
plt.show()


In [ ]:
# -- Gazdag timelapse differences (noisy migrated - noisy migrated_baseline) ------
migrated_gz_diff_noisy = {label: migrated_gz_noisy[label] - migrated_gz_noisy['Baseline'] for label in labels}

fig, axes = plot_migrated_grid(
    migrated_gz_diff_noisy, extent_mig, ncols=3,
    title=f'Gazdag Migration - Noisy TimeLapse Differences (migrated - migrated_baseline)  |  f_c={f_c_GHz} GHz',
)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()

# -- Zoomed ---------------------------------------------------------------------------
fig, axes = plot_migrated_grid(
    migrated_gz_diff_noisy, extent_mig, ncols=3,
    title='Gazdag Migration - Noisy TimeLapse Differences (zoomed)',
)
for ax in np.atleast_1d(axes).ravel()[:len(migrated_gz_diff_noisy)]:
    ax.set_xlim(1.5, 2.5); ax.set_ylim(0.84, 0.62)
_mark_baseline_current(axes, z_depths_all[0], z_depths_all[1:], legend='all')
plt.show()


## Back-Propagation Migration (Noisy)

Same time-reversal approach as the clean-data section above, but the noisy
background-subtracted traces (`outputs_noisy`) are re-injected as sources instead
of the clean ones. Kirchhoff/Gazdag on noisy data is a pure post-processing step
on already-loaded arrays, but back-propagation requires a brand new gprMax
**forward** (FDTD) simulation per scenario, since the excitation file *is* the
time-reversed noisy trace data.

**Sign-bit time reversal:** the first pass (peak-normalised excitation, same as
the clean-data section) showed the Laplace noise spikes acting as their own
competing point sources during back-propagation, interfering at the true source
locations instead of being suppressed by destructive interference. Spatial
focusing during back-propagation is governed almost entirely by *phase*
(zero-crossings), not amplitude — so instead of injecting the peak-normalised
time-reversed wavefield $u(x,\tau)$, we inject only its sign:

$$u_{\text{sign}}(x, \tau) = \operatorname{sign}\big(u(x, \tau)\big)$$

This keeps every zero-crossing and phase trend of the GPR wavelet intact while
squashing the Laplace spikes down to the same $\pm 1$ as the coherent signal —
stripping them of the outsized amplitude that let them dominate the
back-propagated wavefield. `write_backprop_files(..., sign_bit=True)` below
implements this (see `helper_functions/migration.py`); the clean-data section
above is left on the default peak-normalised mode since it has no noise to
suppress.

Files are written to `vertical_timelapse_study/backprop/<slug>_noisy/` — slug
suffixed with `_noisy` to keep them alongside, but distinct from, the clean-data
runs already in `backprop/`.

**This notebook only generates the `.in` files below** — run each through gprMax
externally to produce the `.vti` snapshots the load/plot cells expect (the
clean-data section above has the same split: no cell in this notebook calls
`api()` for back-propagation, snapshots are assumed to already exist on disk). E.g.:

```
python -m gprMax vertical_timelapse_study/backprop/1lambda_noisy/backprop_1lambda_noisy.in
```

Run all 7 `.in` files (`baseline_noisy`, `1lambda_noisy`, ..., `0p03125lambda_noisy`),
then re-run the cells below.

In [ ]:
# -- Back-propagation file generation (noisy) ------------------------------------
# Mirrors the clean-data cell above, but re-injects the noisy background-
# subtracted traces (outputs_noisy) using sign-bit time reversal (sign_bit=True --
# see markdown above) and writes to '<slug>_noisy' folders so the clean-data
# .in/.out/snapshot files are never touched.
slugs_noisy = [f'{slug}_noisy' for slug in slugs_all]

# Same dispersion-limit + edge-source issue as the clean-data cell above (see the
# EDGE_DROP comment there) -- sign-bit traces are if anything sharper, so the same
# low-pass + edge-exclusion treatment is applied here too.
cutoff_hz_noisy = dispersion_limited_cutoff(eps_r=4.0 * eps_r, dx=0.001)

print(f'Back-propagation file generation (noisy, sign-bit)  (stride={STRIDE}, {N_SNAP} snapshots)' + chr(10))
bp_paths_noisy = {}
for label, slug, raw in zip(labels_all, slugs_noisy, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, f'{label} (noisy, sign-bit)', slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN,
        sign_bit=True
    )
    lowpass_filter_excitation(in_path.parent / 'excitation.txt', cutoff_hz_noisy, edge_exclude=EDGE_DROP)
    bp_paths_noisy[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB  |  '
          f'low-pass={cutoff_hz_noisy/1e9:.1f} GHz  |  edge_drop={EDGE_DROP}')
    print(f'           {in_path}')

print(chr(10) + 'NOTE: .in files only -- run each through gprMax externally to produce the '
      '.vti snapshots the cells below expect, then re-run them.')


In [ ]:
# -- Sign-bit time-reversed B-scans + their frequency spectra (noisy) -----------
# Recreates the same time-reversal + sign-bit transform used inside
# write_backprop_files(..., sign_bit=True) above, purely for visualisation:
# row 1 shows the resulting B-scan that gets injected into gprMax as the
# source excitation; row 2 shows its frequency spectrum (mean over traces),
# illustrating the broadband content that lowpass_filter_excitation() trims.
sign_bit_bscans = {}
for label, raw in zip(labels_all, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t)
    data_rev = tapered[::STRIDE, ::-1]
    sign_bit_bscans[label] = np.sign(data_rev)            # (n_src, n_t)

freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

fig, axes = plt.subplots(2, 7, figsize=(22, 8))

sb_data = [(f'{label} (sign-bit)', sb.T) for label, sb in sign_bit_bscans.items()]
plot_bscan_grid(sb_data, x_traces, time_ns, axes=axes[0], vmax=1.0, shared_colorbar=False)

# norm='forward' -- same amplitude-comparability fix applied everywhere else
# this notebook computes an FFT locally.
sb_spectra = [[(f'{label} (sign-bit)',
                np.abs(np.fft.rfft(sb, axis=1, norm='forward')).mean(axis=0))
               for label, sb in sign_bit_bscans.items()]]
plot_spectrum_grid(sb_spectra, freqs_ghz, axes=axes[1], row_colors=('C2',), title=None)
axes[1, 0].set_ylabel('|FFT| (mean over traces)')

plt.suptitle('Sign-Bit Time-Reversed Excitation -- B-scans and Spectra (Noisy)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pyvista

# -- Load focus frame for all 7 noisy datasets (if snapshots exist) --------------
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames_noisy = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_noisy)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} -- run the .in file through gprMax first, skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4,
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames_noisy[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames_noisy[label]["t_actual"]:.3f} ns')

if not focus_frames_noisy:
    print(chr(10) + 'No noisy back-propagation snapshots found yet -- run the .in files '
          'generated above through gprMax, then re-run this cell.')
else:
    y_current_bp_noisy = [y_all[frame['i']] for frame in focus_frames_noisy.values()]
    mag_frames_noisy = {f'{label} (noisy)  |  t={frame["t_actual"]:.2f} ns': frame['mag']
                         for label, frame in focus_frames_noisy.items()}
    ez_frames_noisy  = {f'{label} (noisy)  |  t={frame["t_actual"]:.2f} ns': frame['ez']
                         for label, frame in focus_frames_noisy.items()}

    # -- |E| magnitude -- full extent --------------------------------------------
    fig, axes = plot_wavefield_grid(
        mag_frames_noisy, extent_full, field='magnitude', ncols=4, y_surface=y_surface,
        title=f'Back-Propagation |E| (Noisy) -- All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
    )
    _mark_baseline_current(axes, y_baseline, y_current_bp_noisy, legend='first')
    plt.show()

    # -- |E| magnitude -- zoomed --------------------------------------------------
    fig, axes = plot_wavefield_grid(
        mag_frames_noisy, extent_full, field='magnitude', ncols=4,
        title=f'Back-Propagation |E| (Noisy, zoomed)  |  focus at {t_focus_ns:.2f} ns',
    )
    for ax in np.atleast_1d(axes).ravel()[:len(mag_frames_noisy)]:
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_zoom_lo, y_zoom_hi)
    _mark_baseline_current(axes, y_baseline, y_current_bp_noisy, legend='all')
    plt.show()

    # -- Ez component -- full extent ----------------------------------------------
    fig, axes = plot_wavefield_grid(
        ez_frames_noisy, extent_full, field='signed', ncols=4, y_surface=y_surface,
        title=f'Back-Propagation Ez (Noisy) -- All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
    )
    _mark_baseline_current(axes, y_baseline, y_current_bp_noisy, legend='first')
    plt.show()

    # -- Ez component -- zoomed -----------------------------------------------------
    fig, axes = plot_wavefield_grid(
        ez_frames_noisy, extent_full, field='signed', ncols=4,
        title=f'Back-Propagation Ez (Noisy, zoomed)  |  focus at {t_focus_ns:.2f} ns',
    )
    for ax in np.atleast_1d(axes).ravel()[:len(ez_frames_noisy)]:
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_zoom_lo, y_zoom_hi)
    _mark_baseline_current(axes, y_baseline, y_current_bp_noisy, legend='all')
    plt.show()


In [ ]:
# -- Back-propagation timelapse differences (noisy) (Ez_noisy - Ez_noisy_baseline) --
if not focus_frames_noisy or 'Baseline' not in focus_frames_noisy:
    print('Skipping noisy back-propagation timelapse-difference plot -- '
          'no noisy snapshots (or no Baseline snapshot) yet.')
else:
    bp_ez_diff_noisy = {label: focus_frames_noisy[label]['ez'] - focus_frames_noisy['Baseline']['ez']
                         for label in labels if label in focus_frames_noisy}
    y_current_diff_noisy = y_scatterers_shift[:len(bp_ez_diff_noisy)]

    fig, axes = plot_wavefield_grid(
        bp_ez_diff_noisy, extent_full, field='signed', ncols=3, vmax_headroom=1.0,
        title=f'Back-Propagation (Noisy) -- TimeLapse Differences Ez  |  focus at {t_focus_ns:.2f} ns',
    )
    _mark_baseline_current(axes, y_baseline, y_current_diff_noisy, legend='all')
    plt.show()

    # Zoomed
    fig, axes = plot_wavefield_grid(
        bp_ez_diff_noisy, extent_full, field='signed', ncols=3, vmax_headroom=1.0,
        title='Back-Propagation (Noisy) -- TimeLapse Differences Ez (zoomed)',
    )
    for ax in np.atleast_1d(axes).ravel()[:len(bp_ez_diff_noisy)]:
        ax.set_xlim(x_zoom_lo, x_zoom_hi)
        ax.set_ylim(y_zoom_lo, y_zoom_hi)
    _mark_baseline_current(axes, y_baseline, y_current_diff_noisy, legend='all')
    plt.show()


## Save Migrated Results (Noisy, non-difference)

In [ ]:
# -- Save normal (non-difference) migrated results from the noisy datasets -----
kirchhoff_noisy_all = np.stack([migrated_noisy[lbl].astype(float)    for lbl in labels_all], axis=0)  # (7, n_z, n_x)
gazdag_noisy_all    = np.stack([migrated_gz_noisy[lbl].astype(float) for lbl in labels_all], axis=0)

x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_noisy_planes = []
for lbl in labels_all:
    if lbl in focus_frames_noisy:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames_noisy[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_noisy_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_noisy_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
        print(f'[{lbl}] focus_frames_noisy missing -- filled with NaN')
backprop_noisy_all = np.stack(backprop_noisy_planes, axis=0)   # (7, n_z, n_x)

save_path_mig_noisy = STUDY_ROOT / 'vertical_migrated_results_noisy.npz'
np.savez_compressed(
    save_path_mig_noisy,
    kirchhoff    = kirchhoff_noisy_all,
    gazdag       = gazdag_noisy_all,
    backprop     = backprop_noisy_all,
    scenarios    = np.array(labels_all, dtype='U20'),
    shift_lambda = np.array([0] + shifts_lambda),
    y_scatterers = np.array([y_baseline] + y_scatterers_shift),
    z_depths     = np.array([z_scatterer] + z_depths_shift),
    x_traces     = x_traces,
    z_img        = z_img,
    x_scatterer  = np.float64(x_scatterer),
    y_baseline   = np.float64(y_baseline),
    z_scatterer  = np.float64(z_scatterer),
    z_top        = np.float64(z_top),
    noise_level  = np.float64(NOISE_LEVEL),
)

print(f'Saved -> {save_path_mig_noisy}')
print(f'\nStacked arrays  (n_scenarios=7, n_z={kirchhoff_noisy_all.shape[1]}, n_x={kirchhoff_noisy_all.shape[2]}):')
for name, arr in [('kirchhoff', kirchhoff_noisy_all), ('gazdag', gazdag_noisy_all), ('backprop', backprop_noisy_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f'  ({absent} absent -> NaN)' if absent else ''
    print(f'  {name:<12}  {arr.shape}{note}')
